In [1]:
import pandas as pd
import numpy as np

In [2]:
demo = pd.read_sas("../data/raw/P_DEMO.xpt", format='xport')
ghb = pd.read_sas("../data/raw/P_GHB.xpt", format='xport')

assert demo['SEQN'].is_unique
assert ghb['SEQN'].is_unique

# print(len(demo['SEQN']), len(ghb['SEQN']))

ghb = ghb.rename(columns={'SEQN': 'id'}).astype({'id': int})

ghb

,id,LBXGH
0,109264,5.3
1,109266,5.2
2,109271,5.6
3,109273,5.1
4,109274,5.7
...,...,...
10404,124815,5.8
10405,124817,6.6
10406,124818,5.9
10407,124821,5.9


In [3]:
demo_engineered = demo.copy()

maps = {
    'RIDSTATR': {1: False, 2: True},
    'RIDRETH3': {
        1: 'mexican american',
        2: 'other hispanic',
        3: 'non-hispanic white',
        4: 'non-hispanic black',
        6: 'non-hispanic asian',
        7: 'other/multi-racial'
    },
    'RIDEXMON': {1: 'november_to_april', 2: 'may_to_october'},
    'DMDBORN4': {1: 'usa_born', 2: 'non_usa_born'},
    'DMDYRUSZ': {1: '< 5', 2: '5<=x<15', 3: '15<=x<30', 4: '>=30'},
    'RIAGENDR': {1: 'male', 2: 'female'},
    'SIALANG': {1: 'english', 2: 'spanish'},
    'DMDEDUC2': {
        1: 'less_than_9th_grade',
        2: 'some_high_school',
        3: 'high_school_grad',
        4: 'some_college',
        5: 'college_grad'
    },
    'DMDMARTZ': {
        1: 'married_with_partner',
        2: 'widowed_divorced_separated',
        3: 'never_married'
    },
    'SIAINTRP': {1: True, 2: False},
    'RIDEXPRG': {1: True, 2: False}
}

# Refused / don't know / cannot be determined -> become NaN
missing_codes = {
    'DMDBORN4': {77, 99},
    'DMDYRUSZ': {77, 99},
    'DMDEDUC2': {7, 9},
    'DMDMARTZ': {77, 99},
    'RIDEXPRG': {3}
}

for raw_col, map_dict in maps.items():
    raw_codes = set(demo[raw_col].dropna().unique())
    allowed = set(map_dict) | missing_codes.get(raw_col, set())
    assert raw_codes <= allowed, f"{raw_col}: unexpected codes {raw_codes - allowed}"

    is_bool = all(isinstance(v, bool) for v in map_dict.values())
    demo_engineered[raw_col] = demo[raw_col].map(map_dict).astype('boolean' if is_bool else 'category')
demo_engineered.loc[demo_engineered['WTMECPRP'] < 1e-10, 'WTMECPRP'] = 0

demo_engineered['SEQN'] = demo_engineered['SEQN'].astype('int')
demo_engineered = demo_engineered.rename(columns={
    'SEQN' : 'id',
    'RIAGENDR' : 'sex',
    'RIDSTATR' : 'is_examined',
    'RIDAGEYR' : 'age_at_screening',
    'RIDRETH3' : 'race',
    'RIDEXMON' : 'examination_period',
    'DMDBORN4' : 'birth_place',
    'DMDYRUSZ' : 'years_in_us',
    'SIALANG' : 'language_of_interview',
    'DMDEDUC2' : 'level_of_education',
    'DMDMARTZ' : 'marital_status',
    'SIAINTRP' : 'interpreter_used',
    'RIDEXPRG' : 'is_pregnant',
    'INDFMPIR' : 'income_to_poverty_ratio',
    'WTMECPRP' : 'exam_weight'
}, errors='raise')


In [4]:
demo_engineered = demo_engineered.drop(
    ['SDDSRVYR', 'RIDRETH1', 'FIALANG','FIAPROXY', 
     'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 
     'AIALANGA', 'RIDAGEMN', 'WTINTPRP', 'SIAPROXY'],
    axis=1
)


In [5]:
merged = demo_engineered.merge(
    ghb,
    how='inner',
    on='id',
    validate='one_to_one'
)

merged = merged.rename(columns={
    'LBXGH' : 'hba1c',
    "SDMVPSU" : "survey_psu",
    "SDMVSTRA" : "survey_strata"
}, errors='raise')

assert merged['is_examined'].all()

adults = merged[merged['hba1c'].notna()]
print('with HbA1c:', len(adults))

adults = adults[adults['age_at_screening'] >= 20]
print('aged 20+:', len(adults))

adults = adults[~adults['is_pregnant'].fillna(False)]
print('not pregnant:', len(adults))

display(adults)

with HbA1c: 9737
aged 20+: 8081
not pregnant: 8005


,id,is_examined,sex,age_at_screening,race,examination_period,birth_place,years_in_us,level_of_education,marital_status,is_pregnant,language_of_interview,interpreter_used,exam_weight,survey_psu,survey_strata,income_to_poverty_ratio,hba1c
1,109266,True,female,29.0,non-hispanic asian,may_to_october,non_usa_born,5<=x<15,college_grad,never_married,False,english,False,8154.968193,2.0,168.0,5.00,5.2
2,109271,True,male,49.0,non-hispanic white,may_to_october,usa_born,NaN,some_high_school,never_married,<NA>,english,False,8658.732873,1.0,167.0,NaN,5.6
3,109273,True,male,36.0,non-hispanic white,may_to_october,usa_born,NaN,some_college,never_married,<NA>,english,False,22163.596850,1.0,155.0,0.83,5.1
4,109274,True,male,68.0,other/multi-racial,november_to_april,usa_born,NaN,some_college,never_married,<NA>,english,False,7801.600218,2.0,167.0,1.20,5.7
7,109282,True,male,76.0,non-hispanic white,may_to_october,usa_born,NaN,college_grad,married_with_partner,<NA>,english,False,31816.346500,2.0,164.0,3.61,5.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10404,124815,True,male,52.0,non-hispanic black,may_to_october,usa_born,NaN,some_college,widowed_divorced_separated,<NA>,english,False,27768.450036,1.0,172.0,5.00,5.8
10405,124817,True,female,67.0,mexican american,november_to_april,non_usa_born,>=30,some_high_school,married_with_partner,<NA>,spanish,False,8189.572200,2.0,165.0,2.19,6.6
10406,124818,True,male,40.0,non-hispanic black,november_to_april,usa_born,NaN,college_grad,married_with_partner,<NA>,english,False,21666.889837,1.0,166.0,3.82,5.9
10407,124821,True,male,63.0,non-hispanic black,november_to_april,usa_born,NaN,some_high_school,widowed_divorced_separated,<NA>,english,False,4853.430230,1.0,158.0,3.71,5.9
